# **Semantic Chunking**

1. SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

2. It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [ ]:
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from typing import List, Dict, Any, Optional

# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Langchain specific imports
from langchain.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# load environment variables
load_dotenv()

True

In [ ]:
# sample text
text = """
Langchain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents,  memory and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

# step 1: Split into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

# step 2: Embed each sentences
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("IQ_BASE_URL"),
    model=os.getenv("EMBEDDING_MODEL"),
    dimensions=1536,
)

def cosine_similarity(vec1, vec2):
    """
    cosine similarity measures the angle between two vectors.
    - result close to 1: very similar
    - result close to 0: not related
    - result close to -1: opposite meanings
    """

    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return dot_product/(norm_a*norm_b)

def semantic_search(query, documents, embeddings_models: OpenAIEmbeddings):
    """Simple semantic search implementation"""

    query_embedding = embeddings_models.embed_query(query)
    documents_embeddings = embeddings_models.embed_documents(documents)

    similarities = []
    for idx, document_embedding in enumerate(documents_embeddings):
        similarity = cosine_similarity(document_embedding, query_embedding)
        similarities.append((similarity, documents[idx]))

    # sort by similarity
    similarities.sort(reverse=True)
    return similarities

embedded_sentences = embeddings.embed_documents(sentences)

THRESHOLD = 0.7
chunks = []
current_chunk = [sentences[0]]
for idx in range(1, len(sentences)):
    sim = semantic_search(
        current_chunk[idx - 1],
        [sentences[idx]],
        embeddings,
    )[0][0]

    if sim >= THRESHOLD:
        current_chunk.append(sentences[idx])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[idx]]

chunks.append(" ".join(current_chunk))

print(chunks)